Analyse der Verkehrserreichbarkeit mit Regionalbahnen in Schaidt

Diese Notebook analysiert und visualisiert die Erreichbarkeit der umliegenden bebauten Räume von einer Haltestelle des ÖPNV im Regionalverkehr aus. Als Beispiel ist hier die Haltestelle Schaidt (Pfalz), Wörth am Rhein, Deutschland gewählt. 

Der erste Python-Code dient dazu, räumliche Daten der Untersuchungsregion aus der GeoJSON-Datei einzulesen und diese auf einer interaktiven Karte zu visualisieren. Im Sinne der Übersichtlichkeit wird die Karte als html-Datei in einem Webbrowser geöffnet. 
Zunächst werden erforderliche Bibliotheken importiert, um geographische Vektordaten zu analysieren und Dateien im Webbrowser zu öffnen.
Daraufhin wird die GeoJSON-Datei eingelesen, in einem htlm-Dokument gespeichert und schließlich als interaktive Karte im Webbrowser geöffnet. 
Die räumlichen Raster als Grid sind in der Datei bereits enthalten und begrenzen sich nur auf bewohnte Bereiche der Gemeinden und Städte. Alle Attributinformationen der Ausgangsdaten verbleiben in den Grids, um weitere Analysen mit diesen Daten zu ermöglichen.

In [ ]:
import geopandas as gpd
import webbrowser

"Einlesen der GeoJSON-Datei mit den definierten Grids"
filename = r"C:\Users\Sarah\Documents\Programme Studium\GeoInfo2\SVogel4.github.io\BA\Raster_BZA,Woerth.geojson"

gdf = gpd.read_file(filename)
g = gdf.explore()

file = "Untersuchungsregion.html"
g.save(file)

webbrowser.open(file)

Um die folgenden Analysen durchzuführen, werden representative Punkte für jede Raster-Zelle benötigt. Hierzu werden die Zentroide jedes einzelnen Rasters berechnet und diese anschließend als Punktgeometrien gespeichert. Eine Transformation in das Koordinatenreferenzsystem EPSG 25832, welches das Projektionssystem ETRS89/UTM Zone 32 definiert ist erforderlich. Dies hat den Grund, dass es sich bei der Berechnung der Mittelpunkte um eine geometrische Operation handelt, die im metrischen Koordinatensystem durchgeführt werden solte. Mit der Ausgabe der ersten fünf Zeilen des GeoDataFrames in einer Tabelle wird die Umwandlung in Punktgeometrien geprüft. 

In [2]:
points = gdf.copy()

# 1. CRS zuerst sicherstellen
points = points.to_crs(25832)

# 2. Zentroiden berechnen
points["geometry"] = points.geometry.centroid

# 3. Ergebnis prüfen
points.head()

,fid,rowid,featuretype_name,dataset_name,OBJECTID,id,x_sw,y_sw,x_mp,y_mp,f_staat,f_land,f_wasser,p_staat,p_land,p_wasser,Shape_Length,Shape_Area,ags,geometry
0,1,3538471,DE_Grid_ETRS89-LAEA_100m,de_grid_laea_100m,3538471,100mN28800E41870,4187000.0,2880000.0,4187050.0,2880050.0,10000.0,10000.0,0.0,100.0,100.0,0.0,400.0,10000.0,07334501,POINT (439241.002 5429860.233)
1,2,3538472,DE_Grid_ETRS89-LAEA_100m,de_grid_laea_100m,3538472,100mN28800E41871,4187100.0,2880000.0,4187150.0,2880050.0,10000.0,10000.0,0.0,100.0,100.0,0.0,400.0,10000.0,07334501,POINT (439340.934 5429861.624)
2,3,3538473,DE_Grid_ETRS89-LAEA_100m,de_grid_laea_100m,3538473,100mN28800E41872,4187200.0,2880000.0,4187250.0,2880050.0,10000.0,10000.0,0.0,100.0,100.0,0.0,400.0,10000.0,07334501,POINT (439440.867 5429863.016)
3,4,3538474,DE_Grid_ETRS89-LAEA_100m,de_grid_laea_100m,3538474,100mN28800E41873,4187300.0,2880000.0,4187350.0,2880050.0,10000.0,10000.0,0.0,100.0,100.0,0.0,400.0,10000.0,07334501,POINT (439540.799 5429864.407)
4,5,3538475,DE_Grid_ETRS89-LAEA_100m,de_grid_laea_100m,3538475,100mN28800E41874,4187400.0,2880000.0,4187450.0,2880050.0,10000.0,10000.0,0.0,100.0,100.0,0.0,400.0,10000.0,07334501,POINT (439640.731 5429865.799)


Die berechneten Zentroide werden anschließend in ein für Webkarten geeignetes Koordinatenreferenzsystem transformiert und anschließend ebenfalls interaktiv im Browser dargestellt. 

In [3]:
points = points.to_crs(4326)  # wichtig für Webkarten

# Interaktive Karte erstellen
m = points.explore()

# Karte als HTML-Datei speichern
html_file = "Punkte_Zentroiden.html"
m.save(html_file)

# HTML-Datei im Standardbrowser öffnen
webbrowser.open(html_file)


True

Der folgende Python-Code dient der Erstellung eines integrierten Verkehrsnetzwerks unter Verwendung von Straßendaten aus OpenStreetMap (OSM) und Fahrplandaten im General Transit Feed Specification (GTFS)-Format.
Sowohl die OSM-, als auch die GTFS-Daten sind bereits in entsprechenden Dateien hinterlegt und werden in diesem Schritt geladen. Die OSM-Daten enthalten Informationen über das Straßennetz, verschiedene Wege und weitere Verkehrsinfrastruktur, wohingegen eine GTFS-Datei Fahrpläne und Routeninformationen für Busse liefert. In diesem Fall stammen die GTFS-Daten vom Verkehrsverbund Rhein-Neckar (VRN). 
Für die Erstellung eines integrierten Verkehrsnetzwerks in Schaidt werden die OSM-Daten und die GTFS-Datei kombiniert. Zur Einbindung wird r5py.TransportNetwork aus der Python-Bibliothek r5py benutzt, welches die Grundlage für spätere Erreichbarkeits- und Reisezeitanalysen bildet.

In [4]:
from r5py import TransportNetwork

# Download der OSM-Daten 
osm_fp = r"C:\Users\Sarah\Documents\Programme Studium\GeoInfo2\SVogel4.github.io\BA\OSM_Daten.pbf"
transport_network = TransportNetwork(osm_fp)

# Download der GTFS-Daten
gtfs_fp = r"C:\Users\Sarah\Documents\Programme Studium\GeoInfo2\SVogel4.github.io\BA\Schienenverkehr.zip"

# Erstellen des Verkehrsnetzwerks mit OSM- und GTFS-Daten
transport_network = TransportNetwork(
    # OSM data
    osm_fp, 
    
    # A list of GTFS file(s)
    [gtfs_fp]
)
print("Prepared the network")

Prepared the network


Im nächsten Schritt wird der Ausgangspunkt der Erreichbarkeitsanalyse festgelegt. Hierzu wird die Haltestelle Schaidt, Rathaus geocodiert und ein Punktobjekt zur weiteren Analyse erstellt. 
Zur Geocodierung wird die Bibliothek OSMnx importiert, die auf Daten von OSM zurückgreift. Die Klasse Point aus der Bibliothek Shapely dient zur Erstellung von Punktgeometrien auf Basis von Koordinatenpaaren. 
Die ermittelten Koordinaten der Haltestelle werden in Lat und Lon angegeben, woraus ein Punktobjekt erzeugt wird. Dieses wird in einem GeoDataFrame gespeichert. Als geogrpaisches Referenzsystem wird WGS84 zur Darstellung auf einer Webkarte verwendet. 

In [5]:
import osmnx as ox
from shapely.geometry import Point

# Geokodierung der Adresse "Schaidt (Pfalz)" in Koordinaten
address = "Schaidt (Pfalz)"
lat, lon = ox.geocode(address)

# Erstellen eines GeoDataFrames aus den Koordinaten
origin = gpd.GeoDataFrame({"geometry": [Point(lon, lat)], "name": "Schaidt (Pfalz)", "id": [0]}, index=[0], crs="epsg:4326")

# Interaktive Karte erzeugen
m = origin.explore(
    max_zoom=13,
    color="red",
    marker_kwds={"radius": 12},
    )

# Karte als HTML-Datei speichern
html_file = "Schaidt-PFalz.html"
m.save(html_file)

# Karte im Standardbrowser öffnen
webbrowser.open(html_file)

True

Besonders bei der Analyse mit r5py sind diese Zeilen von Bedeutung, da hiermit eine eindeutige Identifikation der Anfangs- und Endpunkte möglich ist. Es werden fortlaufende Identifikationsnummern erzeugt, wozu für jeden Datensatz eine numerische Kennung erzuegt wurde. Die IDs ermöglichen eine eindeutige Zuordnung der in den netzwerkbasierten Analysen berechneten Reisezeiten zu den jeweiligen Start- und Zielpunkten und werden insbesondere von der Bibliothek r5py für die Erstellung von Reisezeitmatrizen benötigt. 

In [6]:
origin["id"] = range(len(origin))
points["id"] = range(len(points))

Im nächsten Schritt wird die Reisezeitmatrix berechnet. 
Den Startort (origin) stellt die Haltestelle in Schaidt dar, alle restlichen Punkte innerhalb des Untersuchungsgebietes sind die Zielorte (destinations).
Diie Analyse wurde für den 23.06.2026 um 7:15 Uhr durchgeführt, da dies eine typische Pendlerzeit simuliert und um diese Zeit die meisten Schüler in die Schule fahren. 
Als berücksichtige Verkehrsmittel wurden allgemein alle öffentlichen Verkehrsmittel, Regional- und Fernzüge, U-Bahn und Straßenbahnen gewählt.  
Die ersten Zeilen der berechneten Reisezeitmatrix wurden angezeigt, um die funktionsweise der Analyse zu prüfen. 

In [8]:
import datetime
from r5py import TravelTimeMatrix, TransportMode
import geopandas as gpd
import pandas as pd

# Berechnung der Reisezeitmatrix
travel_time_matrix = TravelTimeMatrix(
    transport_network,
    origins=origin,
    destinations=points,
    max_time=datetime.timedelta(minutes=120), # Maximale Reisezeit: 120 Minutens
    departure=datetime.datetime(2026, 6, 23, 7, 10), #Abfahrtszeit: 23. Juni 2026, 07:10 Uhr
    transport_modes=[
        TransportMode.TRANSIT, # Allgemeiner öffentlicher Verkehr
        TransportMode.RAIL,  # Regional- und Fernzüge
        TransportMode.SUBWAY,  # U-Bahn
        TransportMode.TRAM # Straßenbahn
    ],          
    max_time_walking=datetime.timedelta(minutes=7),  
)

travel_time_matrix.head()

,from_id,to_id,travel_time
0,0,0,0.0
1,0,1,NaN
2,0,2,NaN
3,0,3,NaN
4,0,4,NaN


Im nächsten Codeabschnitt wird die berechnete Reisezeitmatrix mit dem Grid verknüpft, um Reisezeiten für die einzelnen Zellen zu ermitteln. Dies ermöglicht die räumliche Visualisierung der Erreichbarkeit auf einer Karte. 
Zuerst wird hierzu ein neuer Index erzeugt, sollte dieser zuvor möglicherweise nicht fortlaufend oder eindeutig gewesen sein. Auch wird der aktuelle Index in eine neue Spalte id kopiert, um eine zuordnung mit der reisezeiten aus der Matrix mit den entsprechenden Polygonflächen zu gewährleisten. 
Anschließend werden die beiden Tabellen zusammengeführt, wozu die Spalten "id" und "to_id" verwendet werden. 
Um den Erfolg der Verknüpfung zu zeigen, werden die ersten fünf Zeilen des Ergebnisses angezeigt. 

In [9]:
gdf = gdf.reset_index(drop=True)
gdf["id"] = gdf.index

# Verknüpfung der Geodaten mit der Reisezeitmatrix
join = gdf.merge(
    travel_time_matrix,
    left_on="id",
    right_on="to_id",
    how="left"
)

join.head()


,fid,rowid,featuretype_name,dataset_name,OBJECTID,id,x_sw,y_sw,x_mp,y_mp,...,p_staat,p_land,p_wasser,Shape_Length,Shape_Area,ags,geometry,from_id,to_id,travel_time
0,1,3538471,DE_Grid_ETRS89-LAEA_100m,de_grid_laea_100m,3538471,0,4187000.0,2880000.0,4187050.0,2880050.0,...,100.0,100.0,0.0,400.0,10000.0,07334501,"POLYGON ((439191.707 5429809.548, 439190.364 5...",0,0,0.0
1,2,3538472,DE_Grid_ETRS89-LAEA_100m,de_grid_laea_100m,3538472,1,4187100.0,2880000.0,4187150.0,2880050.0,...,100.0,100.0,0.0,400.0,10000.0,07334501,"POLYGON ((439291.639 5429810.939, 439290.297 5...",0,1,NaN
2,3,3538473,DE_Grid_ETRS89-LAEA_100m,de_grid_laea_100m,3538473,2,4187200.0,2880000.0,4187250.0,2880050.0,...,100.0,100.0,0.0,400.0,10000.0,07334501,"POLYGON ((439391.572 5429812.331, 439390.229 5...",0,2,NaN
3,4,3538474,DE_Grid_ETRS89-LAEA_100m,de_grid_laea_100m,3538474,3,4187300.0,2880000.0,4187350.0,2880050.0,...,100.0,100.0,0.0,400.0,10000.0,07334501,"POLYGON ((439491.504 5429813.723, 439490.162 5...",0,3,NaN
4,5,3538475,DE_Grid_ETRS89-LAEA_100m,de_grid_laea_100m,3538475,4,4187400.0,2880000.0,4187450.0,2880050.0,...,100.0,100.0,0.0,400.0,10000.0,07334501,"POLYGON ((439591.436 5429815.114, 439590.094 5...",0,4,NaN


In [10]:
valid = join["travel_time"].dropna()

print("Gültige Werte:", valid.shape[0])
print("Fehlende Werte:", join["travel_time"].isna().sum())

print("Durchschnitt (Min):", valid.mean())
print("Median (Min):", valid.median())

Gültige Werte: 211
Fehlende Werte: 1395
Durchschnitt (Min): 68.1611374407583
Median (Min): 74.0


Im letzten Codeabschnitt wird eine interaktive Karte erstellt, die die Reisezeit zu verschiedenen Standorten in der Untersuchungsregion visualisiert. 
Die Darstellung erfolgt anhand einer Choropletenkarte, auf der die einzelnen Zellen basierend auf ihrer Reisezeit zum Startpunkt farblich kodiert werden. Die Farben reichen hierbei von grün (kurze Reisezeit) bis rot (längere Reisezeit). Die Zeitintervalle sind zwischen 0-5 Minuten, 5-10 Minuten, 10-20 Minuten, 20-30 Minuten und über 30 Minuten festgelegt. Die Möglichkeit hierfür bietet die Funktion pd.cut() aus der Bibliothek Pandas.
Zur benutzterfreundlichen Darstellung wird der Kartenmittelpunkt aus allen vorhandenen Rasterzellen berechnet und deren Mittelpunkt festgelegt. Auch die Farben sollen intuitiv zeigen, welche Orte gut und welche weniger gut erreichbar sind.
Damit beim Überfahren einer Fläche mit dem Mauszeiger die zugehörige Reisezeit in Minuten angezeigt wird, werden die klassifizierten Polygonflächen als GeoJSON-Layer in die Karte integriert. 
Die Legende wird manuell mit html und css erstellt, da Folium keine automatische Funktion dazu bereitstellt. 
Die fertige Karte wurde als HTML-Datei gespeichert und zur weiteren Analyse im Webbrowser bereitgestellt.

In [11]:
import folium

# 2. Reisezeit sauber in Minuten
join["travel_min"] = join["travel_time"]


# 3. Klassenbildung (WICHTIG für Legende)
join["travel_class"] = pd.cut(
    join["travel_min"],
    bins=[0, 5, 10, 20, 30, float("inf")],
    labels=["0-5", "5-10", "10-20", "20-30", ">30"],
    include_lowest=True
)

# 4. CRS für webbasierten Kartendienst (Folium)
join_4326 = join.to_crs(4326)


# Gestaltung der Karte 

# 5. Kartenmittelpunkt
center_geom = join_4326.geometry.union_all().centroid
center = [center_geom.y, center_geom.x]

# 6. Farben
colors = {
    "0-5": "#006837",
    "5-10": "#5eaa7f",
    "10-20": "#ffdd57",
    "20-30": "#fd8d3c",
    ">30": "#bd0026"
}


# 7. Style Funktion
def style_function(feature):
    val = feature["properties"]["travel_class"]
    return {
        "fillColor": colors.get(val, "#d9d9d9"),
        "color": "black",
        "weight": 0.2,
        "fillOpacity": 0.7
    }


# 8. Kart
m = folium.Map(
    location=center,
    zoom_start=13,
    tiles="CartoDB positron"
)


# 9. GeoJson Layer
folium.GeoJson(
    join_4326,
    style_function=style_function,
    tooltip=folium.GeoJsonTooltip(
        fields=["travel_min"],
        aliases=["Reisezeit (Min):"]
    )
).add_to(m)


# 10. Legende (fix, manuell)
legend_html = """
<div style="
position: fixed;
bottom: 50px;
left: 50px;
width: 230px;
background: white;
border: 2px solid grey;
z-index: 9999;
font-size: 14px;
padding: 10px;
">

<b>ÖPNV-Reisezeit</b><br>
<b>Abfahrt: 07:10 Uhr</b><br><br>

<div><span style="background:#006837;width:18px;height:18px;display:inline-block;"></span> 0–5 Min</div>
<div><span style="background:#5eaa7f;width:18px;height:18px;display:inline-block;"></span> 5–10 Min</div>
<div><span style="background:#ffdd57;width:18px;height:18px;display:inline-block;"></span> 10–20 Min</div>
<div><span style="background:#fd8d3c;width:18px;height:18px;display:inline-block;"></span> 20–30 Min</div>
<div><span style="background:#bd0026;width:18px;height:18px;display:inline-block;"></span> >30 Min</div>

</div>
"""

m.get_root().html.add_child(folium.Element(legend_html))

# ---------------------------
# 11. Speichern
# ---------------------------
map_path = "karte.html"
m.save(map_path)
webbrowser.open(map_path)

True

In [12]:
# 7. Style Funktion
def style_function(feature):
    val = feature["properties"]["travel_class"]
    return {
        "fillColor": colors.get(val, "#d9d9d9"),
        "color": "black",
        "weight": 0.2,
        "fillOpacity": 0.7
    }


# GeoPackage speichern


join_4326.to_file(
    "Regionalbahn23.gpkg",
    layer="reisezeiten",
    driver="GPKG"
)
print("GeoPackage wurde erfolgreich gespeichert.")

GeoPackage wurde erfolgreich gespeichert.
